# Adversarial Validation — Checking for Train/Test Distribution Shift

---

**Suggested Kaggle discussion post:**

> Before trying pseudo-labeling or importance weighting, I ran adversarial validation to check whether the train and test sets come from the same distribution. The result: AUC = 0.50 — the generator treated both splits identically. This rules out a whole class of approaches (importance reweighting, domain adaptation, test-time augmentation) and confirms the ceiling is purely about signal in the data, not a train/test mismatch. Full walkthrough with code in the notebook below.

---

## What is Adversarial Validation?

In many competitions, the train and test sets are drawn from different time periods, geographies, or collection pipelines. When that happens, a model trained on train data can fail to generalize — not because the model is wrong, but because the *distribution shifted*.

**Adversarial validation** is a simple diagnostic: label train rows as class 0 and test rows as class 1, then train a classifier to tell them apart. The AUC of that classifier tells you how different the two sets are:

| Adversarial AUC | Interpretation |
|---|---|
| ~0.50 | No detectable shift — train and test are i.i.d. |
| 0.55 – 0.65 | Mild shift — worth investigating |
| > 0.70 | Strong shift — major source of generalization error |

When shift is detected, you can use the classifier's probabilities to **importance-weight** training samples — upweighting rows that look like test data and downweighting rows that don't. This is a form of covariate shift correction.

When AUC ≈ 0.50, none of that is needed.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import catboost as cb

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))

FEATURES     = [c for c in train.columns if c not in ['heart_disease', 'id']]
CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']

X      = train[FEATURES]
X_test = test[FEATURES]

print(f'Train: {X.shape[0]:,} rows    Test: {X_test.shape[0]:,} rows')
print(f'Test fraction: {X_test.shape[0] / (X.shape[0] + X_test.shape[0]):.2%}')

## Build the Adversarial Dataset

Combine train and test into one dataset. Label train rows 0, test rows 1.
The task is now: *can a model learn to tell these apart?*

In [ ]:
X_adv = pd.concat([X, X_test], ignore_index=True)
y_adv = np.array([0] * len(X) + [1] * len(X_test))

print(f'Combined dataset: {len(X_adv):,} rows')
print(f'Class balance: {y_adv.mean():.2%} test (expected ~0.30)')

## Train the Adversarial Classifier

We use a shallow CatBoost (depth=4, 200 iterations) — powerful enough to detect shift if it exists, but not so complex it overfits to noise. 5-fold stratified CV gives a robust AUC estimate.

In [ ]:
ADV_PARAMS = dict(
    iterations=200, learning_rate=0.1, depth=4,
    task_type='GPU', cat_features=CAT_FEATURES,
    random_state=42, verbose=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_adv, y_adv)):
    m = cb.CatBoostClassifier(**ADV_PARAMS)
    m.fit(X_adv.iloc[tr_idx], y_adv[tr_idx])
    pred = m.predict_proba(X_adv.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y_adv[val_idx], pred)
    fold_aucs.append(fold_auc)
    print(f'  fold {fold+1}: AUC = {fold_auc:.4f}')

print(f'\nAdversarial AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')

## Result: AUC = 0.50

**0.5018 ± 0.0009** — indistinguishable from random chance.

The classifier cannot tell train from test. This means the synthetic data generator used the same parameters and sampling process for both splits — they are draws from the same distribution. No covariate shift exists.

This rules out:
- Importance weighting / covariate shift correction
- Domain adaptation
- Test-time feature distribution matching

And it confirms: any gap between CV score and leaderboard score is *not* due to distribution mismatch. It's pure sampling variance.

## Feature Importances — A Subtle Point

Even when AUC ≈ 0.50, the adversarial model still reports feature importances. It's tempting to interpret these as "features that differ between train and test" — but that's wrong when AUC is random.

What they actually reflect is **feature variance**: continuous features with many unique values (max_hr, cholesterol, age, bp) get more splits and dominate importance regardless of any actual distributional difference.

In [ ]:
# Train on full combined data to read importances
adv_full = cb.CatBoostClassifier(**ADV_PARAMS)
adv_full.fit(X_adv, y_adv)

importances = pd.Series(adv_full.get_feature_importance(), index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot.barh(ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Adversarial classifier feature importances\n(reflect feature variance, NOT distributional shift — AUC = 0.50)')
ax.set_xlabel('Importance')
ax.axvline(x=100/len(FEATURES), color='red', linestyle='--', alpha=0.5, label='uniform baseline')
ax.legend()
plt.tight_layout()
plt.show()

print('Importances (descending):')
print(importances[::-1].round(1).to_string())

## Confirming with Probability Distributions

A second check: what does the adversarial model predict for the test rows? If it genuinely detected shift, we'd see test-row scores pushed toward 1.0. Instead, we should see scores clustering around the *base rate* — the fraction of the combined dataset that is test.

In [ ]:
test_scores  = adv_full.predict_proba(X_test)[:, 1]
train_scores = adv_full.predict_proba(X)[:, 1]
base_rate    = len(X_test) / (len(X) + len(X_test))  # 0.30

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scores, label, color in [
    (axes[0], train_scores, 'Train rows', 'steelblue'),
    (axes[1], test_scores,  'Test rows',  'orange'),
]:
    ax.hist(scores, bins=60, color=color, alpha=0.8, density=True)
    ax.axvline(scores.mean(), color='black', linewidth=2, label=f'mean = {scores.mean():.3f}')
    ax.axvline(base_rate, color='red', linewidth=1.5, linestyle='--', label=f'base rate = {base_rate:.2f}')
    ax.set_title(f'{label}: P(is from test set)')
    ax.set_xlabel('Adversarial score')
    ax.legend()

plt.tight_layout()
plt.show()

print(f'Base rate (test fraction):  {base_rate:.3f}')
print(f'Mean score on train rows:   {train_scores.mean():.3f}  (expect ≈ {base_rate:.3f})')
print(f'Mean score on test rows:    {test_scores.mean():.3f}  (expect ≈ {base_rate:.3f} if no shift)')

## Why Mean ≈ Base Rate Confirms No Shift

When a classifier is uninformative (AUC = 0.50), its output for any row converges to the **prior probability** of belonging to the positive class — here, 30% (270k test rows / 900k total rows).

Both train and test rows receive scores centered at 0.30 with narrow spread. A useful adversarial classifier would push test-row scores well above 0.5 and train-row scores below. That doesn't happen here.

The distributions are nearly identical — exactly what you'd expect from i.i.d. data.

## Rounding Artifact Analysis

A related question: do features like blood pressure show rounding artifacts (spikes at multiples of 10), and are these artifacts consistent between train and test? If the generator smoothed one split differently, that would show up as shift.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, col in zip(axes, ['bp', 'cholesterol', 'age', 'max_hr']):
    mod_vals = np.arange(10)
    tr_freq = np.array([(X[col].values % 10 == v).mean() for v in mod_vals])
    te_freq = np.array([(X_test[col].values % 10 == v).mean() for v in mod_vals])
    x = np.arange(10)
    ax.bar(x - 0.2, tr_freq, 0.4, label='train', color='steelblue', alpha=0.8)
    ax.bar(x + 0.2, te_freq, 0.4, label='test',  color='orange',    alpha=0.8)
    ax.axhline(0.1, color='red', linestyle='--', linewidth=0.8, label='uniform')
    ax.set_title(col)
    ax.set_xlabel('Last digit (mod 10)')
    ax.set_xticks(x)
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('Digit frequency (mod 10): train vs test — rounding artifacts consistent across splits', y=1.02)
plt.tight_layout()
plt.show()

print('Fraction of values ending in 0 or 5 (rounding artifact):')
print(f'  {"Feature":<15}  {"Train":>7}  {"Test":>7}  {"Diff":>7}  {"Uniform":>8}')
for col in ['bp', 'cholesterol', 'age', 'max_hr']:
    tr = X[col].values
    te = X_test[col].values
    tr_r = ((tr % 10 == 0) | (tr % 10 == 5)).mean()
    te_r = ((te % 10 == 0) | (te % 10 == 5)).mean()
    print(f'  {col:<15}  {tr_r:>7.3f}  {te_r:>7.3f}  {te_r-tr_r:>+7.3f}  {0.2:>8.3f}')

## Summary

| Check | Result | Interpretation |
|---|---|---|
| Adversarial AUC | 0.5018 ± 0.0009 | No distributional shift |
| Mean adversarial score | ≈ base rate (0.30) | Classifier is uninformative |
| Rounding artifacts | Identical in train and test | Generator consistent across splits |

**Conclusion**: The synthetic data generator for this competition draws train and test from the same distribution with identical parameters. There is no train/test mismatch to exploit. The leaderboard ceiling (~0.954) is a property of the signal in the data, not a distributional artifact.